# Revisión de literatura y novedad

Antes de afirmar que un modelo es nuevo hay que buscar con cuidado a quién se le ocurrió primero. Este capítulo
documenta esa búsqueda: dónde buscamos, con qué cadenas, qué encontramos y en qué se diferencia RegionalMAPEM de los
trabajos más cercanos. La conclusión adelantada es que la idea general (combinar clasificadores según la zona del
espacio de clientes) ya existe en churn, y que lo nuevo es la regla con la que nosotras los combinamos.

## Cómo buscamos

| Etapa | Fecha | Fuentes | Qué se hizo |
|---|---|---|---|
| Mapa de modelos usados en churn | 2026-10-30 | Web of Science, Scopus, IEEE Xplore y arXiv | Dos conjuntos por base: A, clasificación con ML con *novel*, *improved*, *enhanced* o *hybrid* en el título, y B, churn con ML, ambos sin redes neuronales; además B2, churn amplio, como control. En Web of Science y Scopus suman 27 313 registros. De ahí salieron ocho modelos candidatos, verificados después en Google Académico y arXiv |
| Búsqueda del mecanismo propio | 2026-11-03 | Web of Science, Scopus, Scopus AI y Google Académico | Cadenas que combinan la definición de churn con cada familia afín: mezcla de expertos, selección dinámica, stacking local y ensambles ponderados, EM y agregación de probabilidades, y calibración. También se repitieron A, B y B2 en Web of Science |
| Índices abiertos y citación | 2026-11-08 | OpenAlex y arXiv | Cadenas adicionales, una de ellas en español, y búsqueda por citación hacia atrás y hacia adelante desde cinco trabajos semilla |

La definición de churn combina el término en inglés con sus sinónimos en inglés y, en Web of Science, en español: *customer
defection*, *customer attrition*, "fuga de clientes", "abandono de clientes" y "deserción de clientes". Las cadenas exactas,
con sus resultados, están en la tabla 4.4.

## Matriz de novedad

La tabla 4.1 resume la búsqueda del mecanismo propio. Las filas de las bases institucionales vienen del registro del 3 de
noviembre, y las de OpenAlex y arXiv, del 8 de noviembre. En las cadenas con muchos resultados se revisó solo una parte: la
columna de revisados dice cuántos.

In [1]:
# Matriz de novedad del modelo propio: registro institucional (2026-11-03) e índices abiertos (2026-11-08)
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT))
from src.presentacion import tabla
LIT = ROOT / "experiments" / "literatura"
leer = lambda n: pd.read_csv(LIT / n, dtype=str, keep_default_na=False)   # los conteos se muestran tal como se registraron

N = leer("matriz_novedad.csv")
PROPIO = N[N["Modelo candidato"].str.contains("propio") & (N["Base de datos"] != "Declaración")]
A = leer("matriz_busqueda_abierta.csv")
filas = pd.concat([
    PROPIO[["Fecha", "Base de datos", "Cadena de búsqueda", "Resultados", "Revisados", "Relevantes", "Observación"]],
    pd.DataFrame({"Fecha": A.fecha, "Base de datos": A.base, "Cadena de búsqueda": A.cadena, "Resultados": A.resultados, "Revisados": A.revisados,
                  "Relevantes": A.candidatos, "Observación": A.titulos_candidatos.replace("", "Sin trabajos relevantes").str.replace(" | ", "; ", regex=False)})])
tabla(filas, "Tabla 4.1. Matriz de novedad del mecanismo propio", decimales=0,
      nota="[churn] es la definición de la tabla 4.4 (fila NOTA-1103). Los identificadores entre paréntesis remiten a esa tabla.")

Fecha,Base de datos,Cadena de búsqueda,Resultados,Revisados,Relevantes,Observación
2026-11-03,WoS,[churn] AND mezcla de expertos / compuerta (W-P1-1103),1,1,0,"Phishing, no churn"
2026-11-03,WoS,[churn] AND selección dinámica / región de competencia (W-P2-1103),1,1,1,"Xiao et al. (2012) DCEID: selección dinámica dura, no pesos MAP-EM"
2026-11-03,WoS,[churn] AND stacking local / ensambles ponderados (W-P3-1103),7,7,4,Solo pesos globales (Khoh 2023; MetaLight-ChurnNet 2026; votación 2023; híbrido 2018)
2026-11-03,WoS,[churn] AND EM / pooling / Dirichlet (W-P4-1103),18,15,0,EM solo en clustering/LDA; ninguno combina clasificadores
2026-11-03,Scopus,[churn] AND mezcla de expertos (S-P1-1103),5,5,1,"Yan et al. (2001) MoE neuronal con compuerta entrenada; difiere en expertos fijos no neuronales, regiones fijas y prior Dirichlet"
2026-11-03,Scopus,[churn] AND selección dinámica (S-P2-1103),3,3,2,"Xiao et al. 2012 y 2014: selección dinámica, no pesos MAP-EM"
2026-11-03,Scopus,[churn] AND stacking local / ensambles ponderados (S-P3-1103),17,17,8,Solo pesos globales
2026-11-03,Scopus,[churn] AND EM / pooling / Dirichlet (S-P4-1103),27,27,1,Lobo et al. (2024) mezcla bayesiana de caducidad (no ensamble)
2026-11-03,Scopus AI,Pregunta en lenguaje natural (SAI-1-1103),9,9,0,"Sin estudios con pesos locales EM/MAP-EM, MoE con expertos fijos ni prior Dirichlet en churn"
2026-11-03,Google Scholar,"Scholar: ""customer churn"" ""mixture of experts"" (G-P1-1103)",97,10,4,Yan 2001 y MMoE (neuronales); LRMoE por EM en seguros (no predice churn)


Ninguna fila contiene el mecanismo completo. En las bases institucionales, lo más cercano es la mezcla de expertos neuronal de
Yan et al. (2001), la selección dinámica de Xiao et al. (2012 y 2014), y una serie de ensambles con pesos globales, entre
ellos el de Khoh et al. (2023). Cuando el EM aparece junto a churn, se usa para segmentar o en modelos de tópicos, no para
combinar clasificadores. Scopus AI, consultado en lenguaje natural, tampoco devolvió estudios con pesos locales estimados
con EM o MAP-EM.

Los antecedentes que obligan a acotar la novedad aparecieron en la cadena de selección dinámica de OpenAlex: dos artículos
sobre churn en telecomunicaciones, de 2010 y 2012, que se discuten abajo.

In [2]:
# Búsqueda por citación desde los trabajos semilla
S = leer("snowballing_resumen.csv")
tabla(pd.DataFrame({"Trabajo semilla": S.semilla, "Año": S.anio, "Citas hacia adelante": S.citas_adelante,
                    "Referencias hacia atrás": S.referencias_atras, "Trabajos sobre churn entre las citas": S.con_churn, "Fecha": S.fecha}),
      "Tabla 4.2. Búsqueda por citación (OpenAlex)", decimales=0, buscar=False)

Trabajo semilla,Año,Citas hacia adelante,Referencias hacia atrás,Trabajos sobre churn entre las citas,Fecha
De Caigny 2018 (logit leaf model),2018,558,77,301,2026-11-08
Soares 2006 (DES-Clustering),2006,75,24,0,2026-11-08
Yan 2001 (MoE en churn),2002,38,13,26,2026-11-08
Sill 2009 (FWLS),2009,262,15,2,2026-11-08
Jordan y Jacobs 1994 (MoE),1994,2779,34,1,2026-11-08


La búsqueda por citación se hizo con OpenAlex porque permite descargar de forma reproducible las referencias y las citas de
cada trabajo (`experiments/literatura/busqueda_citas.py`). Hacia adelante solo se conservaron los trabajos con *churn*,
*attrition* o *retention* en el título o el resumen.

## Los trabajos más cercanos

Comparar solo por aplicación dejaría por fuera a los métodos más parecidos, así que comparamos por familia algorítmica. La
tabla 4.3 reúne diez familias y precisa, para cada una, qué conserva nuestra propuesta y qué cambia.

| Familia y trabajos | ¿Churn? | Qué conserva la propuesta | Qué cambia |
|---|---|---|---|
| **Mezcla de expertos con compuerta EM:** Jacobs et al. (1991); Jordan y Jacobs (1994) {cite}`jordan1994`; Yan et al. (2001); LRMoE, Fung et al. (2019) | Sí, pero neuronal (Yan, 2001; MMoE de Ziadi et al.). LRMoE se usa en tarificación de seguros, no para predecir churn | Mezcla convexa con pesos que dependen de $x$, estimada por EM | La compuerta se fija antes de estimar los pesos y los expertos no se reentrenan. Por eso el objetivo es cóncavo y el EM llega al óptimo global; cuando compuerta y expertos se entrenan juntos, el EM solo garantiza un óptimo local |
| **Selección dinámica:** Woods et al. (1997); Ko et al. (2008); Cruz et al. (2015); DES-Clustering, Soares et al. (2006) {cite}`soares2006`; Xiao et al. (2012, 2014) {cite}`xiao2012` | Sí (Xiao et al.) | La competencia de cada modelo varía en el espacio de entrada; DES-Clustering obtiene regiones por clustering | Selección dura de clasificadores frente a pesos continuos estimados por máxima posterior con un prior de Dirichlet |
| **Integración dinámica por subáreas** (2010 y 2012; dos artículos en chino sobre telecomunicaciones) | Sí | Combinar clasificadores según la zona del espacio de clientes | Regiones duras de k-means, un clasificador nuevo por subárea y pesos por metaheurística sin garantía de óptimo. Aquí las regiones son suaves, los expertos fijos y los pesos salen de MAP-EM con objetivo cóncavo |
| **Modelos por segmento:** logit leaf model, De Caigny et al. (2018) {cite}`decaigny2018`; extensiones (De Caigny et al., 2021; Hendriks, 2022; Idbenjra et al., 2024) | Sí | Tratar distinto a cada segmento de clientes | Segmentos duros de un árbol supervisado, con un modelo nuevo por hoja. Aquí no se entrena nada nuevo: se reponderan benchmarks fijos |
| **Clustering con ensambles globales:** Bilal et al. (2022) | Sí | Combinar clustering y clasificadores | Votación, stacking y AdaBoost globales, sin pesos por clúster ni EM |
| **Stacking y ensambles con pesos globales:** Wolpert (1992); Breiman (1996); Khoh et al. (2023); Jiang et al. (2024) | Sí | Usar predicciones fuera de pliegue para ajustar una combinación ponderada | Un solo vector de pesos, frente a pesos por región con un prior hacia los globales |
| **Stacking con pesos locales:** FWLS, Sill et al. (2009) {cite}`sill2009`; LESTARI, Arifin y Wibowo (2025) | No: LESTARI predice compras de datos prepago | Pesos que varían con la instancia | En FWLS los pesos son lineales en metavariables, sin restricción de símplex, y se ajustan por mínimos cuadrados. Aquí son pesos convexos por región con verosimilitud binaria, prior de Dirichlet y óptimo global |
| **Agregación de probabilidades:** pool lineal, Ranjan y Gneiting (2010); Gneiting y Ranjan (2013) | No encontrado | Combinación lineal de probabilidades | Pesos globales frente a pesos por región aprendidos con MAP-EM |
| **Selección de ensambles centrada en la decisión:** Baumann et al. (2015) | Sí | Combinar modelos optimizando la métrica relevante | Selección global y discreta de miembros, frente a combinación suave por regiones con objetivo de log-verosimilitud |
| **Combinación por regímenes:** Elliott y Timmermann (2005) {cite}`elliott2005` | No (macroeconomía) | Pesos de combinación que cambian según un régimen | Regímenes temporales y pronósticos puntuales con pérdida cuadrática, frente a regiones del espacio de clientes y probabilidades |

**Tabla 4.3.** Diferenciación frente a métodos afines.

Los dos trabajos de 2010 y 2012 son los antecedentes más cercanos. Comparten con nuestra propuesta la intuición de que la
mejor combinación depende de la zona del espacio de clientes. Difieren en el mecanismo: regiones duras, clasificadores
reentrenados por subárea y pesos hallados por metaheurísticas, sin un prior que proteja a las regiones con pocos datos.

## Declaración de novedad

La novedad está en la regla de combinación, no en segmentar clientes ni en integrar clasificadores por zonas, porque
ambas cosas ya existen en churn.

> Al mejor de nuestro conocimiento y según la búsqueda descrita (Web of Science, Scopus, Scopus AI y Google Académico,
> 2026-11-03; OpenAlex y arXiv, con búsqueda por citación desde cinco trabajos semilla, 2026-11-08), no se encontraron
> estudios de predicción de churn que combinen clasificadores fijos y calibrados con pesos que dependen de regiones suaves
> del espacio de clientes, estimados por máxima posterior con EM y un prior de Dirichlet hacia los pesos globales.

Esa regla tiene una consecuencia que no comparte ninguno de los métodos de la tabla: con las regiones fijas, el objetivo es
cóncavo sobre un producto de símplex, así que el EM sube en cada iteración y converge al óptimo global (capítulo 5). Los
resultados de los capítulos 6 y 7 muestran que esa garantía no se traduce en una ventaja predictiva en Telco, pero la novedad
del mecanismo no depende de que gane.

## Alcance de la búsqueda

La revisión se apoya en títulos y resúmenes. Los dos artículos de 2010 y 2012 se analizaron a partir de sus resúmenes, y la
diferenciación de la tabla 4.3 se apoya en lo que esos resúmenes describen. La búsqueda por citación se hizo en OpenAlex, que
no cubre todo lo que cubren Web of Science y Scopus. La ampliación hacia modelos por segmento, DES-Clustering y combinación
por regímenes se hizo por familia de métodos: esos trabajos están en la tabla 4.3, pero sus búsquedas no tienen conteos
registrados y por eso no aparecen en la matriz.

## Registro completo de búsquedas

La tabla 4.4 conserva las cadenas exactas, con la fecha y el número de resultados. La tabla 4.5 muestra la verificación de
los ocho modelos que la primera etapa dejó como candidatos: todos quedaron descartados al buscar en Google Académico, lo que
llevó a proponer un mecanismo propio en lugar de aplicar a churn un modelo existente.

In [3]:
# Registro de búsquedas: cadenas exactas, tal como se ejecutaron
B = leer("registro_busquedas.csv")
tabla(B[["ID", "Base", "Conjunto", "Cadena exacta", "Fecha", "Resultados", "Registros exportados", "Estado", "Observación"]],
      "Tabla 4.4. Registro de búsquedas en bases institucionales y arXiv", decimales=0,
      nota="Las filas con sufijo 1103 son del 2026-11-03. NOTA-1103 define [churn].")

# Verificación de los ocho candidatos de la primera etapa
C = N[~N["Modelo candidato"].str.contains("propio")]
tabla(C[["Fecha", "Base de datos", "Modelo candidato", "Cadena de búsqueda", "Resultados", "Revisados", "Relevantes", "Observación"]],
      "Tabla 4.5. Verificación de los modelos candidatos de la primera etapa", decimales=0)

ID,Base,Conjunto,Cadena exacta,Fecha,Resultados,Registros exportados,Estado,Observación
S0,Scopus,Prueba inicial,"TITLE(churn* OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"")",2026-10-30,3933,,Descartada,"churn* recupera ruido (p. ej. 'Churnam', fitoterapia) y solo busca en título."
W0,WoS,Prueba inicial,"TI=(churn* OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"")",2026-10-30,1527,,Descartada,Primer intento falló ('Failed to fetch') en Fielded Search; en QUERY BUILDER funcionó. Solo prueba de acceso.
W-A,WoS,A: universo clasificación ML,"TS=(""machine learning"") AND TS=(classification OR classifier*) AND TI=(enhanced OR improved OR novel OR hybrid) NOT TS=(""neural network*"" OR ""deep learning"" OR LSTM OR CNN OR convolutional OR recurrent OR transformer* OR autoencoder* OR ""attention mechanism"")",2026-10-30,8759,8759,Exportada,9 lotes de 1.000 (Full Record).
W-B,WoS,B: churn (estilo profesor),"TS=(churn OR churns OR churner* OR churning OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"") AND TS=(predict* OR classif* OR forecast*) AND TS=(""machine learning"") NOT TS=(""neural network*"" OR ""deep learning"" OR LSTM OR CNN OR convolutional OR recurrent OR transformer* OR autoencoder* OR ""attention mechanism"")",2026-10-30,329,329,Exportada,Incluye churn de empleados y de videojuegos.
W-B2,WoS,B2: churn amplio (control),"TS=(churn OR churns OR churner* OR churning OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"") AND TS=(predict* OR classif*) AND TS=(customer* OR subscriber* OR client* OR telecom*)",2026-10-30,978,978,Exportada,75 artículos de W-B no están en W-B2 y viceversa: ambos se usan para descartar.
S-A,Scopus,A: universo clasificación ML,"TITLE-ABS-KEY(""machine learning"") AND TITLE-ABS-KEY(classification OR classifier*) AND TITLE(enhanced OR improved OR novel OR hybrid) AND NOT TITLE-ABS-KEY(""neural network*"" OR ""deep learning"" OR LSTM OR CNN OR convolutional OR recurrent OR transformer* OR autoencoder* OR ""attention mechanism"")",2026-10-30,13074,13074,Exportada,Exportado completo en un CSV.
S-B,Scopus,B: churn (estilo profesor),"TITLE-ABS-KEY(churn OR churns OR churner* OR churning OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"") AND TITLE-ABS-KEY(predict* OR classif* OR forecast*) AND TITLE-ABS-KEY(""machine learning"") AND NOT TITLE-ABS-KEY(""neural network*"" OR ""deep learning"" OR LSTM OR CNN OR convolutional OR recurrent OR transformer* OR autoencoder* OR ""attention mechanism"")",2026-10-30,1157,1157,Exportada,
S-B2,Scopus,B2: churn amplio (control),"TITLE-ABS-KEY(churn OR churns OR churner* OR churning OR ""customer defection"" OR ""customer attrition"" OR ""fuga de clientes"" OR ""abandono de clientes"" OR ""deserción de clientes"") AND TITLE-ABS-KEY(predict* OR classif*) AND TITLE-ABS-KEY(customer* OR subscriber* OR client* OR telecom*)",2026-10-30,3016,3016,Exportada,
I-A,IEEE Xplore,A: clasificación mejorada,"(""All Metadata"":""machine learning"") AND (""All Metadata"":classification OR classifier*) AND (""Document Title"":enhanced OR improved OR novel OR hybrid) NOT (""neural network*"" OR ""deep learning"" OR LSTM OR CNN OR convolutional OR recurrent OR transformer* OR autoencoder* OR ""attention mechanism"")",2026-10-30,6749,6749,Ejecutada,Resúmenes de la página de resultados (truncados a ~400 caracteres); IEEE bloqueó la descarga masiva de resúmenes completos.
I-B,IEEE Xplore,B: churn (estilo profesor),"(churn OR churns OR churner* OR churning OR ""customer defection"" OR ""customer attrition"") AND (predict* OR classif* OR forecast*) AND ""machine learning"" NOT (""neural ne

Fecha,Base de datos,Modelo candidato,Cadena de búsqueda,Resultados,Revisados,Relevantes,Observación
2026-10-30,WoS,Proceso gaussiano,"Conjuntos B y B2 de WoS (hoja Busquedas) + términos: ""gaussian process""",1,1,0,Sin aplicación a churn.
2026-10-30,Scopus,Proceso gaussiano,"Conjuntos B y B2 de Scopus (hoja Busquedas) + términos: ""gaussian process""",2,2,0,Sin aplicación a churn.
2026-10-30,WoS,Twin SVM,"Conjuntos B y B2 de WoS (hoja Busquedas) + términos: ""twin support vector"", TWSVM, TSVM",0,0,0,Sin aplicación a churn.
2026-10-30,Scopus,Twin SVM,"Conjuntos B y B2 de Scopus (hoja Busquedas) + términos: ""twin support vector"", TWSVM, TSVM",0,0,0,Sin aplicación a churn.
2026-10-30,WoS,RVM,"Conjuntos B y B2 de WoS (hoja Busquedas) + términos: ""relevance vector machine"", RVM",0,0,0,Sin aplicación a churn.
2026-10-30,Scopus,RVM,"Conjuntos B y B2 de Scopus (hoja Busquedas) + términos: ""relevance vector machine"", RVM",0,0,0,Sin aplicación a churn.
2026-10-30,WoS,Árboles óptimos (MIP),"Conjuntos B y B2 de WoS (hoja Busquedas) + términos: ""optimal classification tree"", ""optimal decision tree""",0,0,0,Sin aplicación a churn.
2026-10-30,Scopus,Árboles óptimos (MIP),"Conjuntos B y B2 de Scopus (hoja Busquedas) + términos: ""optimal classification tree"", ""optimal decision tree""",0,0,0,Sin aplicación a churn.
2026-10-30,WoS,Árboles oblicuos,"Conjuntos B y B2 de WoS (hoja Busquedas) + términos: ""oblique tree"", ""oblique decision tree""",0,0,0,Sin aplicación a churn.
2026-10-30,Scopus,Árboles oblicuos,"Conjuntos B y B2 de Scopus (hoja Busquedas) + términos: ""oblique tree"", ""oblique decision tree""",0,0,0,Sin aplicación a churn.
